In [ ]:
"""Time series imputation for missing values in the final cybersecurity dataset"""
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
import warnings
import re
warnings.filterwarnings('ignore')

class TimeSeriesImputer:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.date_col = 'Date'
        
    def clean_currency_values(self, series):
        """Clean currency values by removing $, commas, and converting to float"""
        def clean_value(x):
            if pd.isna(x) or x == '' or x == '-':
                return np.nan
            if isinstance(x, (int, float)):
                return float(x)
            
            # Convert to string and clean
            x_str = str(x).strip()
            
            # Remove currency symbols, commas, and other non-numeric characters except decimal point
            x_clean = re.sub(r'[^\d.-]', '', x_str)
            
            # Handle empty strings after cleaning
            if x_clean == '' or x_clean == '-':
                return np.nan
            
            try:
                return float(x_clean)
            except (ValueError, TypeError):
                return np.nan
        
        return series.apply(clean_value)
    
    def load_data(self):
        """Load the CSV file and handle data cleaning"""
        try:
            # First try to load with automatic date parsing
            self.df = pd.read_csv(self.file_path, parse_dates=[self.date_col])
        except Exception as e:
            print(f"Automatic date parsing failed: {e}")
            print("Loading without date parsing and will convert manually...")
            self.df = pd.read_csv(self.file_path)
        
        # Ensure Date column is properly formatted as datetime
        if self.date_col in self.df.columns:
            print(f"Original Date column dtype: {self.df[self.date_col].dtype}")
            
            # Try multiple date format parsing strategies
            if self.df[self.date_col].dtype == 'object':
                date_formats = [
                    '%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y', '%Y/%m/%d',
                    '%b %Y', '%B %Y', '%Y-%m', '%m-%Y', '%b-%Y', '%B-%Y'
                ]
                
                for fmt in date_formats:
                    try:
                        self.df[self.date_col] = pd.to_datetime(self.df[self.date_col], format=fmt)
                        print(f"Successfully parsed dates with format: {fmt}")
                        break
                    except:
                        continue
                else:
                    # If none of the specific formats work, use flexible parsing
                    self.df[self.date_col] = pd.to_datetime(self.df[self.date_col], errors='coerce')
                    print("Used flexible date parsing (some dates might be NaT)")
            
            # Check for any NaT values
            nat_count = self.df[self.date_col].isna().sum()
            if nat_count > 0:
                print(f"Warning: {nat_count} dates could not be parsed and set to NaT")
        
        # Clean all numeric columns (excluding Date column)
        print("Cleaning currency values in numeric columns...")
        numeric_cols = self.df.select_dtypes(include=['object']).columns
        market_price_cols = [col for col in numeric_cols if 'MARKET_PRICE' in col or any(x in col for x in ['DDoS', 'Phishing', 'Ransomware', 'Malware'])]
        
        for col in numeric_cols:
            if col != self.date_col:
                original_non_null = self.df[col].notna().sum()
                self.df[col] = self.clean_currency_values(self.df[col])
                cleaned_non_null = self.df[col].notna().sum()
                if original_non_null != cleaned_non_null:
                    print(f"  Cleaned {col}: {original_non_null} -> {cleaned_non_null} non-null values")
        
        self.df = self.df.sort_values(self.date_col).reset_index(drop=True)
        print(f"Data loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
        if self.date_col in self.df.columns and pd.api.types.is_datetime64_any_dtype(self.df[self.date_col]):
            print(f"Date range: {self.df[self.date_col].min()} to {self.df[self.date_col].max()}")
        return self.df
    
    def analyze_missing_values(self):
        """Analyze the pattern and extent of missing values"""
        total_cells = self.df.shape[0] * self.df.shape[1]
        missing_cells = self.df.isnull().sum().sum()
        missing_percentage = (missing_cells / total_cells) * 100
        
        print(f"Total missing values: {missing_cells} ({missing_percentage:.2f}%)")
        
        # Check which columns have missing values
        missing_columns = self.df.columns[self.df.isnull().any()].tolist()
        print(f"Columns with missing values: {len(missing_columns)}")
        
        # Show top columns with most missing values
        missing_counts = self.df[missing_columns].isnull().sum().sort_values(ascending=False)
        print("\nTop columns with missing values:")
        for col, count in missing_counts.head(10).items():
            print(f"  {col}: {count} missing ({count/len(self.df)*100:.2f}%)")
        
        return missing_columns
    
    def create_time_features(self):
        """Create time-based features to help with imputation"""
        # Check if Date column is datetime
        if self.date_col in self.df.columns and pd.api.types.is_datetime64_any_dtype(self.df[self.date_col]):
            valid_dates = self.df[self.date_col].notna()
            if valid_dates.any():
                self.df.loc[valid_dates, 'year'] = self.df.loc[valid_dates, self.date_col].dt.year
                self.df.loc[valid_dates, 'month'] = self.df.loc[valid_dates, self.date_col].dt.month
                self.df.loc[valid_dates, 'quarter'] = self.df.loc[valid_dates, self.date_col].dt.quarter
        else:
            print("Date column not available or not datetime. Creating sequential time index.")
        
        # Always create time index
        self.df['time_index'] = range(len(self.df))
        
        # If no date features were created, create basic ones
        if 'year' not in self.df.columns:
            # Assume monthly data starting from July 2011
            total_months = len(self.df)
            start_year = 2011
            start_month = 7
            
            years = []
            months = []
            for i in range(total_months):
                year = start_year + (start_month + i - 1) // 12
                month = (start_month + i - 1) % 12 + 1
                years.append(year)
                months.append(month)
            
            self.df['year'] = years
            self.df['month'] = months
            self.df['quarter'] = ((np.array(months) - 1) // 3) + 1
    
    def handle_market_price_data(self, df_imputed):
        """Special handling for market price data with sparse patterns"""
        print("Applying specialised imputation for market price data...")
        
        # Identify market price columns
        market_price_cols = [col for col in df_imputed.columns if 'MARKET_PRICE' in col]
        
        if not market_price_cols:
            print("No market price columns found.")
            return df_imputed
        
        print(f"Found {len(market_price_cols)} market price columns")
        
        for col in market_price_cols:
            if df_imputed[col].isnull().sum() > 0:
                # For market price data, use forward fill first (assuming prices persist)
                df_imputed[col] = df_imputed[col].fillna(method='ffill')
                
                # Then backward fill
                df_imputed[col] = df_imputed[col].fillna(method='bfill')
                
                # For any remaining, use median of non-zero values
                non_zero_vals = df_imputed[col][df_imputed[col] > 0]
                if len(non_zero_vals) > 0:
                    median_val = non_zero_vals.median()
                    df_imputed[col] = df_imputed[col].fillna(median_val)
                else:
                    # If no non-zero values, use overall median
                    df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())
        
        return df_imputed
    
    def simple_imputation(self):
        """Apply simple imputation methods for different scenarios"""
        df_imputed = self.df.copy()
        
        # Identify numeric columns (excluding date and categorical)
        numeric_cols = df_imputed.select_dtypes(include=[np.number]).columns.tolist()
        # Remove time features from numeric columns for imputation
        numeric_cols = [col for col in numeric_cols if col not in ['year', 'month', 'quarter', 'time_index']]
        
        print("Applying simple imputation methods...")
        
        # Special handling for market price data first
        df_imputed = self.handle_market_price_data(df_imputed)
        
        # Forward fill for time series (suitable for consecutive missing values)
        df_imputed[numeric_cols] = df_imputed[numeric_cols].fillna(method='ffill')
        
        # Backward fill for any remaining missing values
        df_imputed[numeric_cols] = df_imputed[numeric_cols].fillna(method='bfill')
        
        # Linear interpolation for time series
        df_imputed[numeric_cols] = df_imputed[numeric_cols].interpolate(
            method='linear', 
            limit_direction='both',
            limit_area='inside'
        )
        
        # For any remaining missing values, use seasonal means
        if 'month' in df_imputed.columns:
            for col in numeric_cols:
                if df_imputed[col].isnull().any():
                    # Group by month and fill with monthly mean
                    monthly_means = df_imputed.groupby('month')[col].transform('mean')
                    df_imputed[col] = df_imputed[col].fillna(monthly_means)
        
        # Final fallback: column median (more robust than mean for price data)
        for col in numeric_cols:
            if df_imputed[col].isnull().any():
                col_median = df_imputed[col].median()
                df_imputed[col] = df_imputed[col].fillna(col_median)
        
        return df_imputed
    
    def advanced_imputation(self, df_partial):
        """Apply advanced machine learning-based imputation"""
        print("Applying advanced imputation methods...")
        
        numeric_cols = df_partial.select_dtypes(include=[np.number]).columns.tolist()
        
        # Remove time features from imputation
        imputation_cols = [col for col in numeric_cols if col not in ['year', 'month', 'quarter', 'time_index']]
        
        # Check if we have enough data for advanced imputation
        if len(imputation_cols) < 2:
            print("Not enough numeric columns for advanced imputation. Using simple methods.")
            return df_partial
        
        # For market price data, use KNN which works well with sparse patterns
        try:
            # Use KNN with small neighbors for price data
            knn_imputer = KNNImputer(n_neighbors=3, weights='distance')
            knn_imputed = knn_imputer.fit_transform(df_partial[imputation_cols])
            df_knn = df_partial.copy()
            df_knn[imputation_cols] = knn_imputed
            print("KNN imputation completed successfully")
            return df_knn
        except Exception as e:
            print(f"KNN imputation failed: {e}")
        
        # Fallback: median imputation (better for price data with outliers)
        print("Falling back to median imputation for remaining values")
        for col in imputation_cols:
            if df_partial[col].isnull().any():
                col_median = df_partial[col].median()
                df_partial[col] = df_partial[col].fillna(col_median)
        
        return df_partial
    
    def country_specific_imputation(self, df_imputed):
        """Apply country-specific logic for imputation"""
        print("Applying country-specific imputation logic...")
        
        # Group columns by country and attack type
        country_codes = ['US', 'GB', 'CA', 'AU', 'UA', 'RU', 'FR', 'DE', 'BR', 'CN', 
                        'JP', 'PK', 'KP', 'KR', 'IN', 'TW', 'NL', 'ES', 'SE', 'MX',
                        'IR', 'IL', 'SA', 'SY', 'FI', 'IE', 'AT', 'NO', 'CH', 'IT',
                        'MY', 'EG', 'TR', 'PT', 'PS', 'AE']
        
        attack_types = ['DDoS', 'Phishing', 'Ransomware', 'Password Attack', 'SQL Injection',
                       'Account Hijacking', 'Defacement', 'Trojan', 'Vulnerability', 'Zero-day',
                       'Advanced persistent threat', 'XSS', 'Malware', 'Data Breach',
                       'Disinformation/Misinformation', 'Targeted Attack', 'Adware',
                       'Brute Force Attack', 'Malvertising', 'Backdoor', 'Botnet',
                       'Cryptojacking', 'Worms', 'Spyware', 'Unknown', 'Others']
        
        # For each country, ensure consistency between individual columns and ALL columns
        for country in country_codes:
            for attack in attack_types:
                country_col = f"{attack}-{country}"
                all_col = f"{attack}-ALL"
                
                if country_col in df_imputed.columns and all_col in df_imputed.columns:
                    # If country data is missing but ALL data exists, use proportional imputation
                    mask = df_imputed[country_col].isnull() & df_imputed[all_col].notnull()
                    if mask.any():
                        # Use historical ratio for better imputation
                        valid_data = df_imputed[(df_imputed[country_col].notnull()) & (df_imputed[all_col].notnull())]
                        if len(valid_data) > 0:
                            ratios = valid_data[country_col] / valid_data[all_col]
                            ratio = ratios.median()  # Use median to avoid outlier influence
                        else:
                            ratio = 1/len(country_codes)
                        
                        df_imputed.loc[mask, country_col] = df_imputed.loc[mask, all_col] * ratio
        
        return df_imputed
    
    def validate_imputation(self, original_df, imputed_df):
        """Validate the imputation results"""
        print("\n" + "="*50)
        print("IMPUTATION VALIDATION REPORT")
        print("="*50)
        
        original_missing = original_df.isnull().sum().sum()
        imputed_missing = imputed_df.isnull().sum().sum()
        
        print(f"Missing values before imputation: {original_missing}")
        print(f"Missing values after imputation: {imputed_missing}")
        
        if original_missing > 0:
            success_rate = (1 - imputed_missing/original_missing) * 100
            print(f"Imputation success rate: {success_rate:.2f}%")
        else:
            print("No missing values found in original data")
        
        # Check data statistics for market price columns
        market_price_cols = [col for col in imputed_df.columns if 'MARKET_PRICE' in col]
        if market_price_cols:
            print(f"\nMarket Price Data Statistics ({len(market_price_cols)} columns):")
            market_data = imputed_df[market_price_cols]
            print(f"  Price range: [${market_data.min().min():.2f}, ${market_data.max().max():.2f}]")
            print(f"  Median price: ${market_data.median().median():.2f}")
            print(f"  Columns with data: {market_data.notna().any().sum()}/{len(market_price_cols)}")
        
        # General statistics
        numeric_cols = imputed_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print(f"\nGeneral Statistics ({len(numeric_cols)} numeric columns):")
            print(f"  Value range: [{imputed_df[numeric_cols].min().min():.2f}, {imputed_df[numeric_cols].max().max():.2f}]")
            print(f"  Median values: {imputed_df[numeric_cols].median().median():.2f}")
        
        return imputed_missing == 0
    
    def save_imputed_data(self, imputed_df, output_path):
        """Save the imputed data to a new CSV file"""
        # Remove temporary columns used for imputation
        columns_to_drop = ['year', 'month', 'quarter', 'time_index']
        final_columns = [col for col in imputed_df.columns if col not in columns_to_drop]
        
        final_df = imputed_df[final_columns]
        final_df.to_csv(output_path, index=False)
        print(f"\nImputed data saved to: {output_path}")
        
        # Also save a summary report
        self.save_imputation_summary(final_df, output_path.replace('.csv', '_summary.txt'))
        
        return final_df
    
    def save_imputation_summary(self, df, summary_path):
        """Save a detailed summary of the imputation results"""
        with open(summary_path, 'w') as f:
            f.write("IMPUTATION SUMMARY REPORT\n")
            f.write("=" * 50 + "\n\n")
            
            f.write(f"Dataset Shape: {df.shape}\n")
            f.write(f"Total Columns: {len(df.columns)}\n")
            
            # Count market price columns
            market_cols = [col for col in df.columns if 'MARKET_PRICE' in col]
            f.write(f"Market Price Columns: {len(market_cols)}\n\n")
            
            # Summary statistics for market price data
            if market_cols:
                f.write("MARKET PRICE DATA SUMMARY:\n")
                f.write("-" * 30 + "\n")
                market_data = df[market_cols]
                
                for col in market_cols[:10]:  # Show first 10 columns
                    col_data = df[col]
                    f.write(f"{col}:\n")
                    f.write(f"  Non-zero values: {len(col_data[col_data > 0])}/{len(col_data)}\n")
                    f.write(f"  Price range: ${col_data.min():.2f} - ${col_data.max():.2f}\n")
                    f.write(f"  Median price: ${col_data.median():.2f}\n\n")
    
    def run_complete_imputation(self, output_path='imputed_cybersecurity_data.csv'):
        """Run the complete imputation pipeline"""
        print("Starting time series imputation pipeline...")
        print("=" * 60)
        
        # Load and clean data
        self.load_data()
        
        # Analyse missing values
        missing_cols = self.analyze_missing_values()
        
        if not missing_cols:
            print("No missing values found! Data is complete.")
            return self.df
        
        # Create time features
        self.create_time_features()
        
        # Apply simple imputation
        df_simple_imputed = self.simple_imputation()
        
        # Check if advanced imputation is needed
        remaining_missing = df_simple_imputed.isnull().sum().sum()
        
        if remaining_missing > 0:
            print(f"\n{remaining_missing} missing values remain after simple imputation.")
            print("Applying advanced imputation...")
            df_advanced_imputed = self.advanced_imputation(df_simple_imputed)
        else:
            df_advanced_imputed = df_simple_imputed
            print("All missing values handled by simple imputation.")
        
        # Apply country-specific logic
        df_final = self.country_specific_imputation(df_advanced_imputed)
        
        # Validate results
        is_complete = self.validate_imputation(self.df, df_final)
        
        # Save results
        final_df = self.save_imputed_data(df_final, output_path)
        
        if is_complete:
            print("Imputation completed successfully! All missing values have been filled.")
        else:
            print(" Imputation completed with some missing values remaining.")
            print("Consider reviewing the data or using alternative imputation methods.")
        
        return final_df

# Debugging function to check your data first
def inspect_data_file(file_path):
    """Inspect the data file to understand its structure"""
    print("Inspecting data file...")
    
    try:
        # Read without parsing to see raw data
        df_raw = pd.read_csv(file_path, nrows=10)
        print("First 10 rows of raw data:")
        print(df_raw.head(10))
        
        print(f"\nData shape: {df_raw.shape}")
        print(f"Columns: {list(df_raw.columns)}")
        
        # Check Date column specifically
        if 'Date' in df_raw.columns:
            print(f"\nDate column sample: {df_raw['Date'].tolist()}")
            print(f"Date column dtype: {df_raw['Date'].dtype}")
        
        # Check market price columns
        market_cols = [col for col in df_raw.columns if 'MARKET_PRICE' in col]
        if market_cols:
            print(f"\nMarket price columns found: {len(market_cols)}")
            print("Sample market price values:")
            for col in market_cols[:5]:  # Show first 5
                sample_vals = df_raw[col].head(3).tolist()
                print(f"  {col}: {sample_vals}")
        
    except Exception as e:
        print(f"Error reading file: {e}")

if __name__ == "__main__":
    # First, inspect your data to understand the issue
    file_path = "CT-0711-1224.csv"  # Replace with your actual file path
    inspect_data_file(file_path)
    
    print("\n" + "="*60)
    print("STARTING IMPUTATION PROCESS")
    print("="*60)
    
    # Initialise the imputer with your CSV file path
    imputer = TimeSeriesImputer(file_path)
    
    # Run the complete imputation pipeline
    imputed_data = imputer.run_complete_imputation('imputed_data_2011-2024.csv')
    
    # Final summary
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    print(f"Original data shape: {imputer.df.shape}")
    print(f"Imputed data shape: {imputed_data.shape}")
    print(f"Total columns processed: {len(imputed_data.columns)}")
    
    # Show sample of imputed market price data
    market_cols = [col for col in imputed_data.columns if 'MARKET_PRICE' in col]
    if market_cols:
        print(f"\nSample of imputed market price data:")
        sample_market = imputed_data[market_cols[:3]].head()  # First 3 market price columns
        print(sample_market)

1. Save CT-0711-1224.csv (the final data file you wish to imputate) in your cwd
2. Run the script

In [ ]:
"""Optional reporting function to visualise imputation results"""
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def create_imputation_report(original_file, imputed_file):
    """Create a visual report comparing original and imputed data"""
    
    # Load both datasets
    original = pd.read_csv(original_file, parse_dates=['Date'])
    imputed = pd.read_csv(imputed_file, parse_dates=['Month'])
    
    # Create comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Missing values heatmap (original)
    plt.subplot(2, 2, 1)
    missing_data = original.isnull()
    sns.heatmap(missing_data, cbar=True, yticklabels=False)
    plt.title('Missing Values Pattern (Original Data)')
    
    # Plot 2: Time series of a sample column before/after imputation
    plt.subplot(2, 2, 2)
    # Find a column with missing values
    sample_col = None
    for col in original.columns[1:10]:  # Check first few columns
        if original[col].isnull().any():
            sample_col = col
            break
    
    if sample_col:
        plt.plot(original['Date'], original[sample_col], 'ro-', alpha=0.7, label='Original (with missing)')
        plt.plot(imputed['Date'], imputed[sample_col], 'bo-', alpha=0.7, label='Imputed')
        plt.title(f'Sample Column: {sample_col}')
        plt.legend()
        plt.xticks(rotation=45)
    
    # Plot 3: Distribution comparison
    plt.subplot(2, 2, 3)
    if sample_col:
        original[sample_col].hist(alpha=0.7, bins=20, label='Original')
        imputed[sample_col].hist(alpha=0.7, bins=20, label='Imputed')
        plt.legend()
        plt.title('Distribution Comparison')
    
    # Plot 4: Missing values by column
    plt.subplot(2, 2, 4)
    missing_by_col = original.isnull().sum()
    missing_by_col = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
    missing_by_col.head(10).plot(kind='bar')
    plt.title('Top 10 Columns with Most Missing Values')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.savefig('imputation_report.png', dpi=300, bbox_inches='tight')
    plt.show()

create_imputation_report('CT-0711-1224.csv', 'imputed_data_2011-2024.csv')

If you would like to visualise the imputation effects, this script can be run to give an overview. 

In [ ]:
"""This is a script to apply SMOTE for potentially imbalanced cybersecurity threat data"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
! pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score


warnings.filterwarnings('ignore')

def load_and_prepare_data(csv_file_path):
    """
    Load and prepare the dataset for SMOTE application
    """
    print("Loading dataset...")
    df = pd.read_csv(csv_file_path)
    
    # Convert Month to datetime and extract time-based features
    df['Month'] = pd.to_datetime(df['Month'])
    df['Year'] = df['Month'].dt.year
    df['Month_num'] = df['Month'].dt.month
    df['Quarter'] = df['Month'].dt.quarter
    
    print(f"Dataset shape: {df.shape}")
    print(f"Date range: {df['Month'].min()} to {df['Month'].max()}")
    
    return df

def detect_imbalanced_threats(df, threshold=0.1, min_samples=10):
    """
    Automatically detect which threat classes are imbalanced
    Returns a list of threat types that need SMOTE application
    """
    print("Detecting imbalanced threat classes...")
    
    # Extract unique threat types from column names
    threat_types = set()
    for col in df.columns:
        if '-' in col and not any(x in col for x in ['Month', 'Year', 'Quarter', 'ALL']):
            threat_type = col.split('-')[0].strip()
            threat_types.add(threat_type)
    
    threat_types = sorted(list(threat_types))
    print(f"Found {len(threat_types)} unique threat types")
    
    imbalance_results = {}
    threats_needing_smote = []
    
    for threat in threat_types:
        # Find ALL columns for each threat type
        all_cols = [col for col in df.columns 
                   if threat in col and 'ALL' in col and '?' not in col]
        
        if all_cols:
            threat_col = all_cols[0]
            threat_data = df[threat_col]
            
            # Convert to binary classification (0 = no threat, 1 = threat present)
            binary_threat = (threat_data > 0).astype(int)
            class_counts = binary_threat.value_counts()
            
            total_samples = len(binary_threat)
            positive_count = class_counts.get(1, 0)
            negative_count = class_counts.get(0, 0)
            
            if positive_count > 0:  # Only consider if there are positive samples
                positive_ratio = positive_count / total_samples
                imbalance_ratio = negative_count / positive_count if positive_count > 0 else float('inf')
                
                imbalance_results[threat] = {
                    'total_samples': total_samples,
                    'positive_count': positive_count,
                    'negative_count': negative_count,
                    'positive_ratio': positive_ratio,
                    'imbalance_ratio': imbalance_ratio,
                    'needs_smote': (positive_ratio < threshold) and (positive_count >= min_samples)
                }
                
                if imbalance_results[threat]['needs_smote']:
                    threats_needing_smote.append(threat)
    
    # Print imbalance analysis
    print("\n" + "="*80)
    print("IMBALANCE DETECTION RESULTS")
    print("="*80)
    print(f"{'Threat Type':<25} {'Positives':<10} {'Negatives':<10} {'Pos Ratio':<12} {'Imbalance':<12} {'Needs SMOTE':<12}")
    print("-"*80)
    
    for threat, stats in sorted(imbalance_results.items(), 
                               key=lambda x: x[1]['positive_ratio']):
        print(f"{threat:<25} {stats['positive_count']:<10} {stats['negative_count']:<10} "
              f"{stats['positive_ratio']:.4f}      {stats['imbalance_ratio']:.2f}x       "
              f"{'YES' if stats['needs_smote'] else 'NO':<12}")
    
    print(f"\nFound {len(threats_needing_smote)} threats needing SMOTE: {threats_needing_smote}")
    
    return threats_needing_smote, imbalance_results

def create_multi_threat_labels(df, threat_type):
    """
    Create binary labels for specific threat types
    """
    # Identify columns related to the specific threat
    threat_columns = [col for col in df.columns if threat_type in col and 'ALL' in col and '?' not in col]
    
    if not threat_columns:
        # Fallback: use country-specific columns
        threat_columns = [col for col in df.columns if threat_type in col and '?' not in col]
    
    print(f"Creating labels for {threat_type} using {len(threat_columns)} columns")
    
    # Create binary labels (1 if any threat activity, 0 otherwise)
    if threat_columns:
        threat_activity = df[threat_columns].sum(axis=1) > 0
        labels = threat_activity.astype(int)
        print(f"{threat_type} distribution: {labels.value_counts().to_dict()}")
    else:
        print(f"No columns found for {threat_type}")
        return None
    
    return labels

def select_features_for_threat(df, threat_labels, exclude_threats=None):
    """
    Select relevant features for the classification task, excluding other threat data
    """
    if exclude_threats is None:
        exclude_threats = []
    
    # Exclude non-feature columns
    exclude_cols = ['Month', 'Year', 'Month_num', 'Quarter']
    
    # Also exclude other threat columns to avoid data leakage
    threat_columns_to_exclude = []
    for threat in exclude_threats:
        threat_cols = [col for col in df.columns if threat in col and col not in exclude_cols]
        threat_columns_to_exclude.extend(threat_cols)
    
    feature_columns = [col for col in df.columns 
                      if col not in exclude_cols + threat_columns_to_exclude]
    
    # Select numerical features only
    numerical_features = df[feature_columns].select_dtypes(include=[np.number]).columns.tolist()
    
    print(f"Selected {len(numerical_features)} features (excluding {len(exclude_threats)} threat types)")
    
    return numerical_features

def apply_smote_to_threat(df, threat_type, features, test_size=0.3, random_state=42):
    """
    Apply SMOTE to a specific threat type
    """
    print(f"\nApplying SMOTE to {threat_type}...")
    
    # Create labels
    labels = create_multi_threat_labels(df, threat_type)
    if labels is None:
        return None
    
    # Prepare data
    X = df[features]
    y = labels
    
    # Handle missing values
    X = X.fillna(X.mean())
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    print(f"Before SMOTE - Class distribution: {pd.Series(y_train).value_counts().to_dict()}")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Apply SMOTE
    smote = SMOTE(random_state=random_state, k_neighbors=min(5, sum(y_train == 1) - 1))
    X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
    
    print(f"After SMOTE - Class distribution: {pd.Series(y_train_smote).value_counts().to_dict()}")
    
    return {
        'threat_type': threat_type,
        'X_train_original': X_train,
        'X_test': X_test,
        'y_train_original': y_train,
        'y_test': y_test,
        'X_train_smote': X_train_smote,
        'y_train_smote': y_train_smote,
        'scaler': scaler,
        'features': features
    }

def evaluate_smote_performance(smote_results, model_type='random_forest'):
    """
    Evaluate the performance improvement after SMOTE
    """
    threat_type = smote_results['threat_type']
    
    if model_type == 'random_forest':
        model = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Train and evaluate on original data
    model.fit(smote_results['X_train_original'], smote_results['y_train_original'])
    y_pred_original = model.predict(smote_results['X_test'])
    
    # Train and evaluate on SMOTE data
    model.fit(smote_results['X_train_smote'], smote_results['y_train_smote'])
    y_pred_smote = model.predict(smote_results['X_test'])
    
    
    # Calculate metrics for original
    f1_original = f1_score(smote_results['y_test'], y_pred_original)
    precision_original = precision_score(smote_results['y_test'], y_pred_original)
    recall_original = recall_score(smote_results['y_test'], y_pred_original)
    
    # Calculate metrics for SMOTE
    f1_smote = f1_score(smote_results['y_test'], y_pred_smote)
    precision_smote = precision_score(smote_results['y_test'], y_pred_smote)
    recall_smote = recall_score(smote_results['y_test'], y_pred_smote)
    
    print(f"\n{threat_type} Performance Comparison:")
    print(f"{'Metric':<15} {'Original':<10} {'SMOTE':<10} {'Improvement':<12}")
    print("-" * 50)
    print(f"{'F1-Score':<15} {f1_original:.4f}     {f1_smote:.4f}     {f1_smote - f1_original:+.4f}")
    print(f"{'Precision':<15} {precision_original:.4f}     {precision_smote:.4f}     {precision_smote - precision_original:+.4f}")
    print(f"{'Recall':<15} {recall_original:.4f}     {recall_smote:.4f}     {recall_smote - recall_original:+.4f}")
    
    return {
        'threat_type': threat_type,
        'original': {'f1': f1_original, 'precision': precision_original, 'recall': recall_original},
        'smote': {'f1': f1_smote, 'precision': precision_smote, 'recall': recall_smote},
        'improvement': {
            'f1': f1_smote - f1_original,
            'precision': precision_smote - precision_original,
            'recall': recall_smote - recall_original
        }
    }

def automated_smote_pipeline(df, imbalance_threshold=0.1, max_threats=5):
    """
    Automated pipeline to detect imbalanced threats and apply SMOTE
    """
    print("="*80)
    print("AUTOMATED SMOTE PIPELINE FOR IMBALANCED THREATS")
    print("="*80)
    
    # Detect imbalanced threats
    threats_needing_smote, imbalance_results = detect_imbalanced_threats(
        df, threshold=imbalance_threshold
    )
    
    if not threats_needing_smote:
        print("No threats need SMOTE application based on the current threshold.")
        return {}
    
    # Limit to top N most imbalanced threats
    threats_to_process = threats_needing_smote[:max_threats]
    print(f"\nProcessing top {len(threats_to_process)} most imbalanced threats: {threats_to_process}")
    
    # Apply SMOTE to each threat
    smote_results = {}
    performance_comparisons = {}
    
    for threat in threats_to_process:
        try:
            # Select features excluding other threats being processed
            other_threats = [t for t in threats_to_process if t != threat]
            features = select_features_for_threat(df, None, exclude_threats=other_threats)
            
            # Apply SMOTE
            result = apply_smote_to_threat(df, threat, features)
            
            if result is not None:
                smote_results[threat] = result
                
                # Evaluate performance
                perf = evaluate_smote_performance(result)
                performance_comparisons[threat] = perf
                
        except Exception as e:
            print(f"Error processing {threat}: {e}")
            continue
    
    # Generate summary report
    print("\n" + "="*80)
    print("SMOTE APPLICATION SUMMARY")
    print("="*80)
    
    summary_data = []
    for threat, perf in performance_comparisons.items():
        summary_data.append({
            'Threat': threat,
            'Original_F1': perf['original']['f1'],
            'SMOTE_F1': perf['smote']['f1'],
            'F1_Improvement': perf['improvement']['f1'],
            'Recall_Improvement': perf['improvement']['recall']
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    return {
        'smote_results': smote_results,
        'performance_comparisons': performance_comparisons,
        'summary': summary_df,
        'imbalance_analysis': imbalance_results
    }

def create_smote_augmented_dataset(df, smote_results):
    """
    Create an augmented dataset with SMOTE-generated samples
    """
    print("\nCreating SMOTE-augmented dataset...")
    
    # Start with original dataset
    augmented_df = df.copy()
    
    # For each threat that had SMOTE applied, add synthetic samples
    synthetic_samples = []
    
    for threat, results in smote_results.items():
        # Get the synthetic samples
        X_synthetic = results['X_train_smote'][len(results['X_train_original']):]
        y_synthetic = results['y_train_smote'][len(results['y_train_original']):]
        
        # Create new rows for synthetic samples
        for i, (X_row, y_val) in enumerate(zip(X_synthetic, y_synthetic)):
            if y_val == 1:  # Only add synthetic positive samples
                # Create a new row
                new_row = {}
                
                # Add feature values
                for j, feature in enumerate(results['features']):
                    new_row[feature] = X_row[j]
                
                # Set the threat-specific columns
                threat_cols = [col for col in df.columns if threat in col]
                for col in threat_cols:
                    if 'ALL' in col:
                        # Set appropriate value for the threat
                        new_row[col] = 1  # Synthetic threat occurrence
                    else:
                        # For country-specific columns, you might want more sophisticated logic
                        new_row[col] = 0  # Or use average/median values
                
                # Set other columns to reasonable defaults
                new_row['Month'] = df['Month'].max()  # Most recent month
                new_row['Year'] = df['Year'].max()
                new_row['Month_num'] = (df['Month_num'].max() % 12) + 1
                new_row['Quarter'] = (df['Quarter'].max() % 4) + 1
                
                synthetic_samples.append(new_row)
    
    # Add synthetic samples to the dataset
    if synthetic_samples:
        synthetic_df = pd.DataFrame(synthetic_samples)
        augmented_df = pd.concat([augmented_df, synthetic_df], ignore_index=True)
        print(f"Added {len(synthetic_samples)} synthetic samples to the dataset")
        print(f"Original dataset: {len(df)} rows")
        print(f"Augmented dataset: {len(augmented_df)} rows")
    else:
        print("No synthetic samples were generated")
    
    return augmented_df

def advanced_imbalance_analysis(df):
    """
    Advanced analysis of class imbalance across different dimensions
    """
    print("\n" + "="*80)
    print("ADVANCED IMBALANCE ANALYSIS")
    print("="*80)
    
    # Analyze by country
    countries = ['US', 'GB', 'CA', 'AU', 'UA', 'RU', 'FR', 'DE', 'BR', 'CN', 
                        'JP', 'PK', 'KP', 'KR', 'IN', 'TW', 'NL', 'ES', 'SE', 'MX',
                        'IR', 'IL', 'SA', 'SY', 'FI', 'IE', 'AT', 'NO', 'CH', 'IT',
                        'MY', 'EG', 'TR', 'PT', 'PS', 'AE']
    
    print("\nCountry-wise Threat Analysis:")
    print(f"{'Country':<8} {'Total Threats':<15} {'Avg Threat Rate':<15}")
    print("-" * 45)
    
    country_stats = {}
    for country in countries:
        country_cols = [col for col in df.columns if f"-{country}" in col and 'ALL' not in col]
        if country_cols:
            threat_rates = []
            for col in country_cols:
                threat_rate = (df[col] > 0).mean()
                threat_rates.append(threat_rate)
            
            avg_threat_rate = np.mean(threat_rates)
            country_stats[country] = {
                'total_threats': len(country_cols),
                'avg_threat_rate': avg_threat_rate
            }
            print(f"{country:<8} {len(country_cols):<15} {avg_threat_rate:.4f}        ")
    
    # Temporal analysis
    print("\nTemporal Analysis (by Year):")
    yearly_stats = df.groupby('Year').size()
    print(yearly_stats)
    
    return country_stats

def main():
    """
    Main function to run the automated SMOTE pipeline
    """
    # Configuration
    CSV_FILE_PATH = "imputed_cybersecurity_data.csv"  # Update with your file path
    IMBALANCE_THRESHOLD = 0.1  # Threats with positive ratio < 10% are considered imbalanced
    MAX_THREATS_TO_PROCESS = 20  # Limit number of threats to process
    
    print("AUTOMATED SMOTE FOR IMBALANCED CYBER THREATS")
    print("="*60)
    
    # Load data
    df = load_and_prepare_data(CSV_FILE_PATH)
    
    # Run automated SMOTE pipeline
    pipeline_results = automated_smote_pipeline(
        df, 
        imbalance_threshold=IMBALANCE_THRESHOLD,
        max_threats=MAX_THREATS_TO_PROCESS
    )
    
    if not pipeline_results:
        print("No threats were processed. Exiting.")
        return
    
    # Create augmented dataset
    augmented_df = create_smote_augmented_dataset(df, pipeline_results['smote_results'])
    
    # Advanced analysis
    advanced_stats = advanced_imbalance_analysis(df)
    
    # Save results
    output_files = {}
    
    # Save augmented dataset
    augmented_filename = "cyber_threats_smote_augmented.csv"
    augmented_df.to_csv(augmented_filename, index=False)
    output_files['augmented_dataset'] = augmented_filename
    
    #Advanced stats
    
    advanced_stats_filename = "advanced_imbalance_analysis.csv"
    advanced_stats_df = pd.DataFrame.from_dict(advanced_stats, orient='index')
    advanced_stats_df.to_csv(advanced_stats_filename, index=False)
    output_files['advanced_imbalance_analysis'] = advanced_stats_filename
    
    # Save performance summary
    summary_filename = "smote_performance_summary.csv"
    pipeline_results['summary'].to_csv(summary_filename, index=False)
    output_files['performance_summary'] = summary_filename
    
    # Save imbalance analysis
    imbalance_df = pd.DataFrame.from_dict(pipeline_results['imbalance_analysis'], orient='index')
    imbalance_filename = "threat_imbalance_analysis.csv"
    imbalance_df.to_csv(imbalance_filename)
    output_files['imbalance_analysis'] = imbalance_filename
    
    print(f"\nOutput files created:")
    for file_type, filename in output_files.items():
        print(f"  - {file_type}: {filename}")
    
    return {
        'original_data': df,
        'augmented_data': augmented_df,
        'pipeline_results': pipeline_results,
        'output_files': output_files
    }

# Functions to include for optional plotting and parameter tuning
def plot_imbalance_analysis(imbalance_results):
    """
    Plot the imbalance analysis results (requires matplotlib)
    """
    try:
        
        
        threats = list(imbalance_results.keys())
        positive_ratios = [imbalance_results[t]['positive_ratio'] for t in threats]
        needs_smote = [imbalance_results[t]['needs_smote'] for t in threats]
        
        plt.figure(figsize=(12, 8))
        colors = ['red' if needs else 'blue' for needs in needs_smote]
        plt.barh(threats, positive_ratios, color=colors)
        plt.axvline(x=0.1, color='red', linestyle='--', label='Imbalance Threshold (10%)')
        plt.xlabel('Positive Class Ratio')
        plt.title('Threat Class Imbalance Analysis')
        plt.legend()
        plt.tight_layout()
        plt.show()
        
    except ImportError:
        print("Matplotlib not available for plotting")

def smote_parameter_tuning(df, threat_type, features):
    """
    Tune SMOTE parameters for optimal performance
    """
    
    
    labels = create_multi_threat_labels(df, threat_type)
    X = df[features].fillna(df[features].mean())
    y = labels
    
    # Create pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', RandomForestClassifier(random_state=42))
    ])
    
    # Define parameter grid
    param_grid = {
        'smote__k_neighbors': [3, 5, 7],
        'smote__sampling_strategy': ['auto', 0.5, 0.8],
        'classifier__n_estimators': [50, 100]
    }
    
    # Perform grid search
    grid_search = GridSearchCV(
        pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1
    )
    
    grid_search.fit(X, y)
    
    print(f"Best parameters for {threat_type}: {grid_search.best_params_}")
    print(f"Best F1 score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

if __name__ == "__main__":
    # Run the main automated pipeline
    results = main()
    
    print("\n" + "="*80)
    print("SMOTE APPLICATION COMPLETED SUCCESSFULLY!")
    print("="*80)

1. Save the "imputed_cybersecurity_data.csv" file  (the final data file you wish to imputate) in your cwd
2. Run the script

In [ ]:
"""Exponential Smoothing and Double Exponential Smoothing for Time Series Data"""

from matplotlib import pyplot as plt
import numpy as np
import csv

def exponential_smoothing(series, alpha):

    result = [series[0]] # first value is same as series
    for n in range(1, len(series)):
        result.append(alpha * series[n] + (1 - alpha) * result[n-1])
    return result
  
def plot_exponential_smoothing(series, alphas, attack):
 
    plt.figure(figsize=(17, 8))
    for alpha in alphas:
        plt.plot(exponential_smoothing(series, alpha), label="Alpha {}".format(alpha))
    plt.plot(series.values, "c", label = "Actual")
    plt.legend(loc="best")
    plt.axis('tight')
    plt.title("Exponential Smoothing - "+attack)
    plt.grid(True);

def double_exponential_smoothing(series, alpha, beta):

    result = [series[0]]
    for n in range(1, len(series)+1):
        if n == 1:
            level, trend = series[0], series[1] - series[0]
        if n >= len(series): # forecasting
            value = result[-1]
        else:
            value = series[n]
        last_level, level = level, alpha * value + (1 - alpha) * (level + trend)
        trend = beta * (level - last_level) + (1 - beta) * trend
        result.append(level + trend)
    return result

def plot_double_exponential_smoothing(series, alphas, betas, attack):
     
    plt.figure(figsize=(17, 8))
    for alpha in alphas:
        for beta in betas:
            plt.plot(double_exponential_smoothing(series, alpha, beta), label="Alpha {}, beta {}".format(alpha, beta))
    plt.plot(series, label = "Actual")
    plt.legend(loc="best")
    plt.axis('tight')
    plt.title("Double Exponential Smoothing - "+attack)
    plt.grid(True)
    plt.show()

#The below script performs double exponential smoothing for the data
alpha=float(0.1)
beta=float(0.3)
file_name='imputed_cybersecurity_data_for smoothing.txt'
fin = open(file_name)
rawdat = np.loadtxt(fin, delimiter='\t')
print(rawdat)
print(rawdat.shape)


smoothed=[]
i=0
for r in rawdat.transpose():
    smoothed.append(double_exponential_smoothing(r, alpha,beta))
smoothed = list(map(list, zip(*smoothed)))


with open("sm_data.csv", "w",newline="") as f:
    writer = csv.writer(f)
    writer.writerows(smoothed [:-1]) #do not include the last extra row


